# МТС · Qwen3-0.6B +500 · итоговая проверка 383 звонков

**Обучения здесь нет.** Один FP16-прогон всех 383 официальных звонков, затем неизменённый evaluator v2.1. Нужна GPU (предпочтительно A100) и один ZIP `mts_qwen06_official383.zip`. Полный +500 адаптер уже включён, базовая Qwen скачивается с закреплённой revision.

Запускайте 1 → 2 → 3. Результаты каждого звонка сохраняются на Drive; завершённые звонки повторно не генерируются при продолжении того же запуска. Старое обучение и его бюджет не изменяются. Это отдельная проверка максимум на 4 часа, включая подготовку, с остановкой не позже 07:00 МСК. Прохождение порогов заранее не обещается.


## 1. Подключить GPU/Drive и создать отдельный бюджет проверки
Если прежняя проверка на той же VM ещё работает, сначала дождитесь её завершения или остановите вручную. Не запускайте два GPU-процесса одновременно.


In [ ]:
import json,time,sys,subprocess,os,hashlib,zipfile,re
from pathlib import Path
from datetime import datetime,timezone
from google.colab import drive,files
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],check=True)
drive.mount('/content/drive')
TOTAL_MINUTES = 240
ABSOLUTE_DEADLINE_MSK = '2026-11-16T07:00:00+03:00'
RESUME_RUN = ''  # Если нужно продолжить, сюда тот же путь RUN с Drive.
assert 15 <= TOTAL_MINUTES <= 240
RUNS=Path('/content/drive/MyDrive/MTS_Qwen06_Official383_eval');RUNS.mkdir(exist_ok=True)
locator=RUNS/'last_run.json'
if RESUME_RUN:RUN=Path(RESUME_RUN)
elif locator.is_file():RUN=Path(json.loads(locator.read_text())['run'])
else:RUN=RUNS/datetime.now(timezone.utc).strftime('official383_%Y%m%dT%H%M%SZ')
RUN.mkdir(parents=True,exist_ok=True)
locator.write_text(json.dumps({'run':str(RUN)}))
budget_path=RUN/'eval_budget.json'
if not budget_path.exists():
    now=time.time();end=min(now+TOTAL_MINUTES*60,datetime.fromisoformat(ABSOLUTE_DEADLINE_MSK).timestamp())
    budget={'origin_epoch':now,'deadline_epoch':end,'total_minutes':TOTAL_MINUTES,
            'absolute_deadline':ABSOLUTE_DEADLINE_MSK,'scope':'Independent inference-only evaluation; old training budget untouched'}
    budget_path.write_text(json.dumps(budget,indent=2))
else:
    budget=json.loads(budget_path.read_text())
    assert budget['total_minutes']==TOTAL_MINUTES and budget['absolute_deadline']==ABSOLUTE_DEADLINE_MSK,'Preserve saved budget settings'
assert budget['deadline_epoch']-time.time()>300,'Бюджет этой проверки исчерпан; автоматически не сбрасываем'
print('RUN:',RUN)
print('Осталось минут:',round((budget['deadline_epoch']-time.time())/60,1))


## 2. Загрузить один ZIP и подготовить библиотеки
Torch/CUDA не переустанавливаются. Проверяются SHA-256 всех файлов, выбранного адаптера и official val.


In [ ]:
ROOT=Path('/content/mts_qwen06_official383')
PAYLOAD_NAME='mts_qwen06_official383.zip'
PAYLOAD_SHA256='2a33c38a1958b40f2f1c21da49123ac7ff39f6056dc96b5f16999808d7e28c99'
payload=Path('/content')/PAYLOAD_NAME
if not payload.is_file() or hashlib.sha256(payload.read_bytes()).hexdigest()!=PAYLOAD_SHA256:
    uploaded=files.upload()
    found=False
    for name,data in uploaded.items():
        canonical=re.sub(r' \(\d+\)(?=\.zip$)','',Path(name).name)
        if canonical==PAYLOAD_NAME:
            assert hashlib.sha256(data).hexdigest()==PAYLOAD_SHA256,'ZIP другой версии или повреждён'
            payload.write_bytes(data);found=True
    del uploaded
    assert found,'Загрузите mts_qwen06_official383.zip'
assert hashlib.sha256(payload.read_bytes()).hexdigest()==PAYLOAD_SHA256
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(payload) as z:
    assert len(z.namelist())==len(set(z.namelist())) and sum(i.file_size for i in z.infolist())<200*1024**2
    for name in z.namelist():
        assert (ROOT/name).resolve().is_relative_to(ROOT.resolve()) and chr(92) not in name and ':' not in name
    assert z.testzip() is None
    z.extractall(ROOT)
manifest=json.loads((ROOT/'package_manifest.json').read_text())
for relative,want in manifest['sha256'].items():
    p=(ROOT/relative).resolve()
    assert p.is_relative_to(ROOT.resolve()) and hashlib.sha256(p.read_bytes()).hexdigest()==want,relative
assert not list(ROOT.glob('**/train*.jsonl')),'Training data unexpectedly included'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements.txt')],check=True)
print('Готовы сохранённый +500 и 383 официальных звонка. Обучения не будет.')


## 3. Запустить итоговую проверку
Появятся строки `official_val383: 1/383 ... remaining_est_min=...`. Каждая строка — проверенный звонок. До первой строки идёт загрузка модели. При повторном запуске читаются уже сохранённые записи. Промежуточный ZIP скачивается и при остановке; неполная проверка явно помечается PARTIAL.


In [ ]:
env=os.environ.copy();env['PYTHONPATH']=str(ROOT);env['TOKENIZERS_PARALLELISM']='false';env['PYTHONUNBUFFERED']='1'
finished=False
if (RUN/'final_report.json').is_file():
    finished=json.loads((RUN/'final_report.json').read_text()).get('complete') is True
try:
    if finished:print('Все 383 звонка уже проверены; повторной генерации не будет.')
    else:
        remaining=budget['deadline_epoch']-time.time()
        assert remaining>300,'Нет остатка бюджета'
        result=subprocess.run([sys.executable,'-u',str(ROOT/'eval_only.py'),'--root',str(ROOT),'--run',str(RUN)],
            cwd=ROOT,env=env,timeout=remaining)
        print('Код завершения:',result.returncode)
except subprocess.TimeoutExpired:print('Время этапа исчерпано; записи звонков сохранены на Drive. Полноту смотрим в отчёте.')
finally:
    subprocess.run([sys.executable,str(ROOT/'pack_run.py'),str(RUN)],cwd=ROOT,env=env,check=True)
    result_zip=RUN.parent/(RUN.name+'_results.zip')
    files.download(str(result_zip))
print('Папка результатов:',RUN)


## 4. Посмотреть результат или повторно скачать ZIP
Нужен `complete=true` и `n_completed=383`. `failed` остаётся ошибкой генерации, даже если для штрафного подсчёта метрик экспортируется пустой список фактов. Сохранены сырые ответы, источники, ошибки и неизменённые метрики.


In [ ]:
for name in ['final_report.json','failure.json','official_val383/progress.json']:
    p=RUN/name
    if p.is_file():print(name,p.read_text(encoding='utf-8'))
subprocess.run([sys.executable,str(ROOT/'pack_run.py'),str(RUN)],cwd=ROOT,check=True)
files.download(str(RUN.parent/(RUN.name+'_results.zip')))
